In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from feast import FeatureStore

# Run notebook from correct_phoenix/ (parent of feature_repo/)
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "feature_repo" / "feature_store.yaml").exists():
    FEAST_REPO = REPO_ROOT / "feature_repo"
elif (REPO_ROOT.parent / "feature_repo" / "feature_store.yaml").exists():
    FEAST_REPO = REPO_ROOT.parent / "feature_repo"
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError("Run this notebook from correct_phoenix/")

# Load MinIO/S3 credentials (WSL-safe; avoids CRLF issues from .env)
_env_file = FEAST_REPO / ".env"
if _env_file.exists():
    for line in _env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip().strip("\r")
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key, value)

FEATURE_NAMES = json.loads(
    (FEAST_REPO / "artifacts" / "credit_fe_feature_names.json").read_text()
)

# Required so Feast ODFV can import credit_feature_engineering when run from notebook
import sys

if str(FEAST_REPO) not in sys.path:
    sys.path.insert(0, str(FEAST_REPO))

print(f"Feast repo: {FEAST_REPO.resolve()}")
print(f"Engineered features: {len(FEATURE_NAMES)}")

Feast repo: /mnt/c/Users/guslc/FeatureStore/feature_store_s3/correct_phoenix/feature_repo
Engineered features: 29


In [2]:
store = FeatureStore(repo_path=str(FEAST_REPO))

# All contract entities (same ids materialized into Redis)
emprestimos = pd.read_parquet(FEAST_REPO / "data/historico_emprestimos.parquet")
entity_rows = [{"id_contrato": int(x)} for x in emprestimos["id_contrato"].unique()]

BATCH_SIZE = 1000  # tune if Redis / ODFV struggles
print(f"Entities to fetch: {len(entity_rows):,}")

Entities to fetch: 186,890


In [ ]:
from tqdm.auto import tqdm

feature_service = store.get_feature_service("credit_scoring")
frames = []

for start in tqdm(range(0, len(entity_rows), BATCH_SIZE), desc="Online features"):
    chunk = entity_rows[start : start + BATCH_SIZE]
    batch_df = store.get_online_features(
        entity_rows=chunk,
        features=feature_service,
    ).to_df()
    frames.append(batch_df)

all_engineered = pd.concat(frames, ignore_index=True)
all_engineered = all_engineered[["id_contrato"] + [c for c in FEATURE_NAMES if c in all_engineered.columns]]

print(f"Retrieved {len(all_engineered):,} rows x {len(FEATURE_NAMES)} features")
all_engineered[].head()


/home/guslc/miniconda3/envs/feast/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/guslc/miniconda3/envs/feast/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()
Online features:  69%|██████▉   | 129/187 [01:00<00:28,  2.06it/s]/home/guslc/miniconda3/envs/feast/lib/python3.12/site-packages/botocore/auth.py:422: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime_now = datetime.datetime.utcnow()
On

Retrieved 186,890 rows x 29 features


,id_contrato,valor_solicitado,valor_credito,valor_bem,valor_parcela,valor_entrada,percentual_entrada,qtd_parcelas_planejadas,taxa_juros_padrao,taxa_juros_promocional,...,feat_tipo_contrato_enc,feat_status_contrato_enc,feat_tipo_pagamento_enc,feat_finalidade_emprestimo_enc,feat_tipo_cliente_enc,feat_tipo_portfolio_enc,feat_tipo_produto_enc,feat_categoria_bem_enc,feat_setor_vendedor_enc,feat_canal_venda_enc
0,2802425,607500.0,679671.0,607500.0,25188.615,NaN,NaN,36.0,NaN,NaN,...,0.0,0.0,3.0,24.0,2.0,2.0,2.0,26.0,10.0,3.0
1,2330894,148500.0,174361.5,148500.0,12165.210,NaN,NaN,24.0,NaN,NaN,...,0.0,0.0,0.0,24.0,2.0,2.0,2.0,26.0,10.0,5.0
2,1182516,405000.0,451777.5,405000.0,20361.600,NaN,NaN,30.0,NaN,NaN,...,0.0,0.0,0.0,24.0,2.0,2.0,2.0,26.0,10.0,5.0
3,1543131,229500.0,241920.0,229500.0,22619.520,NaN,NaN,12.0,NaN,NaN,...,0.0,0.0,0.0,24.0,2.0,2.0,2.0,26.0,10.0,5.0
4,2261993,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,1.0,3.0,23.0,2.0,4.0,0.0,26.0,10.0,5.0


In [ ]:
# Full matrix: one row per contract, columns = model features
features_matrix = all_engineered.set_index("id_contrato")[FEATURE_NAMES]

features_matrix.shape

In [ ]:
# Batch retrieval
ids = [2802425, 1594684, 1000019]
batch = store.get_online_features(
    entity_rows=[{"id_contrato": i} for i in ids],
    features=store.get_feature_service("credit_scoring"),
).to_df()

batch.set_index("id_contrato")[FEATURE_NAMES]